# Class defining general configurations for the XAI explainers

In [1]:
# Defs for 
#        T-Explainer,
#        SHAP Explainer
#        KernelSHAP
#        SHAP ExactExplainer
#        LIME
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion

In [ ]:
import sys

!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow
!{sys.executable} -m pip install captum
!{sys.executable} -m pip install shap

In [3]:
import numpy as np
import pandas as pd
import nbimporter

# Utils
import torch
import os
import pickle
from sklearn.base import clone

import xgboost as xgb
from sklearn.neural_network import MLPClassifier

import lime
import lime.lime_tabular

import shap
shap.initjs()

from captum.attr import IntegratedGradients
from captum.attr import InputXGradient
from captum.attr import DeepLift
from captum.attr import LRP
from captum.attr import NoiseTunnel
from captum.attr import Saliency
from captum.attr import GuidedBackprop
from captum.attr import Occlusion
from captum.attr import Lime, LimeBase
from captum.attr import KernelShap

import Taylor_Explainer as texp

In [21]:
class Explainers:
    
    'Class defining general configurations for the XAI explainers we are using.'
        
    #methods from captum library
    #vanilla gradient
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def vnl_grad(self, model, x):

        vnGd= Saliency(model)
        x.requires_grad_()
        vnGd_x_exp= (vnGd.attribute(x.unsqueeze(0), abs=False)).squeeze().detach()
        
        return vnGd_x_exp
    
    
    #integrated gradients
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def int_grad(self, model, x):
        
        itGd= IntegratedGradients(model)
        x.requires_grad_()
        itGd_x_exp= (itGd.attribute(x)).squeeze().detach()
        
        return itGd_x_exp
    

    #input x gradient
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def inx_grad(self, model, x):
        
        iXGd= InputXGradient(model)
        x.requires_grad_()
        iXGd_x_exp= (iXGd.attribute(x.unsqueeze(0))).squeeze().detach()
        
        return iXGd_x_exp
    
    
    #deepLIFT
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def dp_lift(self, model, x):
        dLif= DeepLift(model)
        x.requires_grad_()
        dLif_x_exp= (dLif.attribute(x.unsqueeze(0))).squeeze().detach()
        
        return dLif_x_exp
    
    
    #LRP
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def lrp(self, model, x):
        lwrp= LRP(model)
        x.requires_grad_()
        lwrp_x_exp= (lwrp.attribute(x.unsqueeze(0))).squeeze().detach()
        
        return lwrp_x_exp

    
    #smoothgrad
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # if method=True it uses Vanilla Gradient, if method=False it uses Integrated Gradients
    # RETURN importances as a tensor([i1, ..., in])
    def smo_grad(self, model, x, method:bool=True):

        x.requires_grad_()
        
        if method: 
            nt= NoiseTunnel(Saliency(model))
            smoo_x_exp= (nt.attribute(x.unsqueeze(0), nt_type='smoothgrad', stdevs=0.05, nt_samples=10,
                                      abs=False)).squeeze().detach()
        else: 
            nt= NoiseTunnel(IntegratedGradients(model))
            smoo_x_exp= (nt.attribute(x.unsqueeze(0), nt_type='smoothgrad', stdevs=0.05, nt_samples=10, 
                                      baselines=torch.zeros(x.shape),
                                      return_convergence_delta=False)).squeeze().detach()

        return smoo_x_exp

    
    #guided backpropagation
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def g_bkprop(self, model, x):

        gdBp= GuidedBackprop(model)
        x.requires_grad_()
        gdBp_x_exp= (gdBp.attribute(x.unsqueeze(0))).squeeze().detach()
        
        return gdBp_x_exp

    
    #occlusion
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def occ(self, model, x):
        
        occl= Occlusion(model)
        occl_x_exp= (occl.attribute(x.unsqueeze(0), sliding_window_shapes=(1,1), 
                                    perturbations_per_eval=3)).squeeze().detach()
        
        return occl_x_exp

    
    #lime
    # model is a PyTorch model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def c_lime(self, model, x):
        
        lm= Lime(model)
        lime_x_exp= (lm.attribute(x.unsqueeze(0), n_samples=100)).squeeze().detach()
        
        return lime_x_exp

    
    #kernelSHAP
    # model is a PyTorch model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def c_kshap(self, model, x):
        
        kshap= KernelShap(model)
        kshap_x_exp= (kshap.attribute(x.unsqueeze(0), n_samples=100)).squeeze().detach()
        
        return kshap_x_exp
    
    
    #method from LIME library
    #bring explanations into data order (since LIME automatically orders according to highest importance)
    def lime_exp_in_data_order(self, lime_exp, num_fts):

        exp= np.zeros(num_fts)

        for k, v in lime_exp.local_exp[1]:
            exp[k]= v

        return exp
    
    
    #lime tabular explainer
    # model is a scikit-learn binary classifier model
    #   data is a (m, n) dimensional Pandas DataFrame -- model's training data
    # labels is a (m, 1) dimensional Pandas DataFrame -- training data labels
    #      x is a (1, n) dimensional Pandas DataFrame -- the instance under explanation
    # RETURN importances as a importances as a tensor([i1, ..., in])
    def lime(self, model, data, labels, x):
    
        lime_exp_gen= lime.lime_tabular.LimeTabularExplainer(training_data=np.asarray(data), 
                                                             feature_names=np.asarray(data.columns), 
                                                             training_labels=labels.values.ravel().astype(int), 
                                                             class_names=np.asarray([0,1]), 
                                                             mode='classification', 
                                                             discretize_continuous=False, random_state=1234, 
                                                             verbose=False)

        lime_scores= lime_exp_gen.explain_instance(data_row=np.asarray(x)[0], 
                                                   predict_fn=model.predict_proba, 
                                                   num_features=data.shape[1])

        lime_x_exp= torch.from_numpy(self.lime_exp_in_data_order(lime_scores, data.shape[1]))
        
        return lime_x_exp
    
    
    #methods from SHAP library
    #TreeSHAP
    # model is a XGBModel tree-based binary classifier
    #   data is a (m, n) dimensional Pandas DataFrame -- model's training data
    #      x is a (1, n) dimensional Pandas DataFrame -- the instance under explanation
    # RETURN importances as a importances as a tensor([i1, ..., in])
    def t_shap(self, model, x):
        
        if not isinstance(model, xgb.XGBModel):
            raise ValueError('model should be a XGBModel tree-based binary classifier!')

        shap_exp_gen= shap.TreeExplainer(model)
        shap_x_exp= shap_exp_gen(x)
        shap_x_exp= (torch.from_numpy(shap_x_exp.values)).squeeze()

        return shap_x_exp
            
        
    #SHAP Explainer
    # model is a scikit-learn binary classifier model
    #   data is a (m, n) dimensional Pandas DataFrame -- model's training data
    #      x is a (1, n) dimensional Pandas DataFrame -- the instance under explanation
    # RETURN importances as a importances as a tensor([i1, ..., in])
    def shap(self, model, data, x):
        
        shap_exp_gen= shap.Explainer(model.predict, data)
        shap_x_exp= shap_exp_gen(x)
        shap_x_exp= (torch.from_numpy(shap_x_exp.values)).squeeze()
        
        return shap_x_exp
        
        
    #KernelSHAP
    # model is a scikit-learn binary classifier model
    #   data is a (m, n) dimensional Pandas DataFrame -- model's training data
    #      x is a (1, n) dimensional Pandas DataFrame -- the instance under explanation
    # RETURN importances as a importances as a tensor([i1, ..., in])
    def k_shap(self, model, data, x):
        
        if (data.shape[0]> 100): K_size= 100
        else: K_size= data.shape[0]
        
        shap_krnel_exp_gen= shap.KernelExplainer(model.predict, shap.sample(data, K_size))
        #shap_krnel_x_exp= shap_krnel_exp_gen(x)
        shap_krnel_x_exp= shap_krnel_exp_gen.shap_values(x, silent=True)
        shap_krnel_x_exp= (torch.from_numpy(shap_krnel_x_exp)).squeeze()
        
        return shap_krnel_x_exp
        
    
    #ExactSHAP
    # model is a scikit-learn binary classifier model
    #   data is a (m, n) dimensional Pandas DataFrame -- model's training data
    #      x is a (1, n) dimensional Pandas DataFrame -- the instance under explanation
    # RETURN importances as a importances as a tensor([i1, ..., in])
    def e_shap(self, model, data, x):
        
        if (data.shape[1]> 16):
            raise ValueError('ExactSHAP is for models with less than ~16 features!')
            
        shap_exact_exp_gen= shap.explainers.Exact(model.predict, data)
        shap_exact_x_exp= shap_exact_exp_gen(x)
        shap_exact_x_exp= (torch.from_numpy(shap_exact_x_exp.values)).squeeze()
        
        return shap_exact_x_exp
    
    
    #method from T-Explainer library
    # model is a scikit-learn binary classifier model
    #   data is a (m, n) dimensional Pandas DataFrame -- model's training data
    # labels is a (m, 1) dimensional Pandas DataFrame -- training data labels
    #      x is a (1, n) dimensional Pandas DataFrame -- the instance under explanation
    #      y is a (1, 1) dimensional Pandas DataFrame -- the label of the instance under explanation
    # descriptor defines the parameters to explanations
    # cat_fts list indicating the categorical columns. if empty it considers all features as numeric
    # train_data is the model's training data requered by T-Explainer only for categorical cases
    # labels_train is the model's training labels data requered by T-Explainer only for categorical cases
    # RETURN importances as a importances as a tensor([i1, ..., in])
    def t_exp(self, model, data, labels, x, y, descriptor, cat_fts=[], train_data=[], labels_train= []):
        
        if (np.asarray(cat_fts).shape[0]> 0):
            
            t_x_exp, t_x_ft, t_x_sh= texp.categorical_taylor_explainer(model, data, labels, 
                                              x, y, cat_cols=cat_fts,
                                              h_min=descriptor['h_min'], h_max=descriptor['h_max'],
                                              eps=descriptor['jacobian_eps'], max_itr=descriptor['max_itr'],
                                              finite_diff_version=2, delta=descriptor['ohe_delta'], 
                                              e_x=True, angle=False, retrained_ohe_model=True, 
                                              verbose=False)
        else:
            t_x_exp, t_x_ft, t_x_sh= texp.taylor_explainer(model, data, labels, x, y,
                                              h_min=descriptor['h_min'], h_max=descriptor['h_max'],
                                              eps=descriptor['jacobian_eps'], max_itr=descriptor['max_itr'],
                                              finite_diff_version=2, e_x=True, angle=False, verbose=False)
        
        return torch.from_numpy(t_x_exp)
        

# Tests

In [5]:
import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

synth_ox= pd.read_csv('data/synth_OX_20.csv')

# split synth into features (x) and target (y)
df_inputs= synth_ox.loc[:,synth_ox.columns[0:20]]
df_labels= synth_ox.loc[:,synth_ox.columns[20:21]]

# split df_inputs and df_labels into train (80%) and test (20%) datasets
train_ox, test_ox, labels_train_ox, labels_test_ox= sklearn.model_selection.train_test_split(df_inputs,
                                                                                             df_labels,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)

In [6]:
# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ox= MLPClassifier(activation='relu', alpha=0.0001, hidden_layer_sizes=(64, 64, 64), 
                            learning_rate_init=0.01, max_iter=500, random_state=0, solver='sgd')

nn3_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn3_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn3_model_ox.predict(test_ox))
acc_nn3_ox

0.835

In [8]:
# Create the XGBoost classifier 
xgb_model= xgb.XGBRFClassifier(learning_rate= 0.01, n_estimators= 500, max_depth= 6,
                               gamma= 0.1, subsample= 0.9,
                               objective= 'binary:logistic',
                               eval_metric='logloss')

xgb_model.fit(train_ox, labels_train_ox.values.ravel())

acc_xgb= sklearn.metrics.accuracy_score(labels_test_ox, xgb_model.predict(test_ox))
acc_xgb

0.81

In [7]:
nn_pytorch_model_ox= texp.sklearn_to_pytorch_NN(nn3_model_ox, train_ox.shape[1])

target_i= pd.DataFrame(data=[train_ox.iloc[0,:]], columns=train_ox.columns)
target_l= pd.DataFrame(data=[labels_train_ox.iloc[0]], columns=labels_train_ox.columns)
x_data_tensor= torch.tensor(np.asarray(target_i), dtype=torch.float32)

In [10]:
exp= Explainers()

In [136]:
exp.dp_lift(nn_pytorch_model_ox, x_data_tensor)

tensor([-6.6575,  1.8752, -6.0496,  4.5113,  0.1354,  1.4713, -0.0338, -0.4076,
         1.3699, -2.8737,  5.1172,  1.0209,  0.3882,  1.8144,  0.0570, -0.9116,
        -0.5277, -0.9357,  6.2458,  0.6547])

In [141]:
exp.g_bkprop(nn_pytorch_model_ox, x_data_tensor)

tensor([ -9.4376,   7.4746, -15.3771,  12.7608,   0.4124,   3.9255,  -0.1084,
         -3.1206,   4.8622,  -7.7922,   6.7460,   1.6791,   0.8806,   3.7649,
          0.2166,  -2.6158,  -4.1351,  -2.2249,  16.3407,   1.8816])

In [121]:
exp.int_grad(nn_pytorch_model_ox, x_data_tensor)

tensor([ 2.3118e+00, -1.2443e+00, -4.4929e+00,  4.0081e+00, -9.9089e-01,
         1.0025e+00,  3.3317e+00,  1.1684e+00, -6.9247e-01, -7.9798e+00,
         3.2478e-03, -2.2521e+00,  1.7643e-01, -4.6122e-01, -2.9627e-01,
         5.0165e-01, -3.3814e-01,  4.5120e-01,  9.8310e-01, -1.7543e-01],
       dtype=torch.float64)

In [122]:
exp.inx_grad(nn_pytorch_model_ox, x_data_tensor)

tensor([-6.6575,  1.8752, -6.0496,  4.5113,  0.1354,  1.4713, -0.0338, -0.4076,
         1.3699, -2.8737,  5.1172,  1.0209,  0.3882,  1.8144,  0.0570, -0.9116,
        -0.5277, -0.9357,  6.2458,  0.6547])

In [123]:
exp.c_kshap(nn_pytorch_model_ox, x_data_tensor)

tensor([-0.3791, -1.1136, -3.0072,  2.7740, -0.7492,  0.1837,  1.5997,  0.8160,
         0.1705, -6.4345,  0.2966, -0.8746,  0.7708, -0.3567, -0.1685,  0.1053,
        -1.4764,  0.4047,  1.4725,  1.1241])

In [124]:
exp.c_lime(nn_pytorch_model_ox, x_data_tensor)

tensor([ 0.7375, -1.0359, -3.5115,  2.7355, -0.3607,  0.2059,  2.2887,  0.7276,
        -0.3112, -4.8963,  1.3092, -1.4593, -0.3850, -0.5103, -0.8324, -0.2085,
        -0.7529, -0.4132,  1.6117,  0.2276])

In [125]:
exp.lrp(nn_pytorch_model_ox, x_data_tensor)

tensor([-6.6575,  1.8752, -6.0496,  4.5113,  0.1354,  1.4713, -0.0338, -0.4076,
         1.3699, -2.8737,  5.1172,  1.0209,  0.3882,  1.8144,  0.0570, -0.9116,
        -0.5277, -0.9357,  6.2458,  0.6547])

In [126]:
exp.occ(nn_pytorch_model_ox, x_data_tensor)

tensor([-3.2708,  1.0653, -4.8717,  2.0690, -0.9839,  1.3859, -0.2458, -0.6021,
         1.1064, -6.1630,  0.7070, -0.9935, -0.1417,  1.1071,  0.4104, -1.0770,
        -0.7904, -0.9089,  2.9651,  0.2560])

In [127]:
exp.vnl_grad(nn_pytorch_model_ox, x_data_tensor)

tensor([ -9.4376,   7.4746, -15.3771,  12.7608,   0.4124,   3.9255,  -0.1084,
         -3.1206,   4.8622,  -7.7922,   6.7460,   1.6791,   0.8806,   3.7649,
          0.2166,  -2.6158,  -4.1351,  -2.2249,  16.3407,   1.8816])

In [128]:
exp.smo_grad(nn_pytorch_model_ox, x_data_tensor)

tensor([ -6.8557,   5.3428, -13.0386,   9.9803,  -1.4617,   3.7640,   0.0171,
         -1.5212,   3.7305,  -7.7789,   4.8409,   0.6691,   0.5954,   3.1800,
          1.0831,  -0.5127,  -4.8712,  -1.0128,  12.1970,   1.3488])

In [129]:
exp.smo_grad(nn_pytorch_model_ox, x_data_tensor, False)

tensor([ 2.7842, -1.2979, -4.3186,  4.3647, -1.0999,  0.7419,  3.1654,  1.4164,
        -0.7152, -7.8917,  0.5113, -2.3734,  0.0113, -0.5159, -0.2058,  0.4636,
        -0.3524,  0.3325,  1.0438, -0.1667], dtype=torch.float64)

In [41]:
exp.lime(nn3_model_ox, train_ox, labels_train_ox, target_i)

X does not have valid feature names, but MLPClassifier was fitted with feature names


tensor([ 0.0013,  0.0016, -0.1568,  0.1927, -0.0193, -0.0437,  0.0310,  0.0586,
         0.0060, -0.1147,  0.0477, -0.0026, -0.0062,  0.0039,  0.0386, -0.0202,
        -0.0291, -0.0254,  0.0747,  0.0230], dtype=torch.float64)

In [42]:
exp.t_shap(xgb_model, target_i)

tensor([ 8.1221e-05, -1.1966e-04,  3.5812e-04, -3.5528e-03, -2.5720e-05,
         1.0914e-03,  7.2974e-06, -2.1223e-03, -1.0508e-04, -4.7569e-04,
         4.4980e-04,  1.8565e-06, -1.4452e-05,  5.4147e-06, -1.3907e-03,
         1.8184e-05,  8.9437e-06,  2.0449e-05, -2.4990e-04,  1.6633e-06])

In [43]:
exp.shap(nn3_model_ox, train_ox, target_i)

tensor([-0.1988, -0.0525, -0.0638, -0.0988,  0.0096,  0.0163, -0.0067, -0.1083,
        -0.0242, -0.0417,  0.1167,  0.0096, -0.0042,  0.0021, -0.0550, -0.0121,
         0.1229,  0.0029, -0.0979, -0.0162], dtype=torch.float64)

In [24]:
exp.k_shap(nn3_model_ox, train_ox, target_i)

tensor([-0.1563, -0.0478, -0.0729, -0.1140,  0.0093,  0.0067, -0.0024, -0.1032,
        -0.0294, -0.0408,  0.1088,  0.0000, -0.0036,  0.0080, -0.0642, -0.0052,
         0.1233,  0.0089, -0.1073, -0.0180], dtype=torch.float64)

In [47]:
descriptor_ox= dict()

h_min_dist_ox= texp.get_minimum_distance(train_ox)

# T-Exp explanation settings
descriptor_ox['h_min']= h_min_dist_ox
descriptor_ox['h_max']= 1
descriptor_ox['jacobian_eps']= 1e-3
descriptor_ox['max_itr']= 30
descriptor_ox['ohe_delta']= 0

In [45]:
exp.t_exp(nn3_model_ox, train_ox, labels_train_ox, target_i, target_l, descriptor_ox)

tensor([-1.3673,  0.9321, -2.2972,  2.0480, -0.2611,  0.1591,  0.0327,  0.4525,
         0.7817, -1.7835,  0.9285, -0.2456,  0.1305,  0.9396,  0.4843,  0.1872,
        -0.1427, -0.0776,  2.4009,  0.2505], dtype=torch.float64)